# Sesión 02 - Lab 2: Ingesta de datos semi-estructurados (JSON anidado)

Este laboratorio lee un archivo JSON con pedidos de e-commerce, donde cada registro trae un `struct` anidado (`cliente`, con una `direccion` anidada dentro) y un array de `struct` (`items`, uno por producto del pedido). El objetivo es aplanarlo hacia una tabla gobernada por Unity Catalog, con un grano definido explícitamente. Antes de correrlo, sube `pedidos_ecommerce.json` al volume `/Volumes/dbassociate/default/vol_landing/sesion_02/`.

## Verificación del entorno

In [ ]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_02")


## Lab 2A: Leer el JSON anidado con schema explícito

El archivo está en formato JSON Lines (un objeto por línea). Igual que con CSV, se define un `StructType` explícito en vez de dejar que Spark infiera el schema: para un JSON con estructura anidada, la inferencia automática puede fallar en distinguir un campo opcional ausente de un error de formato.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, DateType, ArrayType
)
from pyspark.sql.functions import col

schema_direccion = StructType([
    StructField("calle", StringType(), True),
    StructField("ciudad", StringType(), True),
    StructField("region", StringType(), True),
    StructField("pais", StringType(), True),
])

schema_cliente = StructType([
    StructField("cliente_id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("email", StringType(), True),
    StructField("direccion", schema_direccion, True),
])

schema_item = StructType([
    StructField("producto", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DoubleType(), True),
])

schema_pedido_json = StructType([
    StructField("pedido_id", IntegerType(), False),
    StructField("fecha_pedido", DateType(), False),
    StructField("canal", StringType(), True),
    StructField("cliente", schema_cliente, True),
    StructField("items", ArrayType(schema_item), True),
])

path_json = "/Volumes/dbassociate/default/vol_landing/sesion_02/pedidos_ecommerce.json"

df_pedidos_raw = (
    spark.read.schema(schema_pedido_json).json(path_json)
    .withColumn("source_file", col("_metadata.file_name"))
)

df_pedidos_raw.printSchema()
print("Pedidos leídos:", df_pedidos_raw.count())


## Lab 2B: Aplanar el struct anidado con notación de punto

`cliente.direccion.ciudad` accede directamente a un campo dos niveles adentro del struct, sin necesidad de un `explode()` (los structs no son arrays: cada pedido tiene un único cliente, no una lista de clientes). `source_file` se calcula una sola vez, apenas se lee el archivo, porque `_metadata` no sobrevive a un `select()` que no la incluya explícitamente.

In [ ]:
df_pedidos_cliente = df_pedidos_raw.select(
    "pedido_id",
    "fecha_pedido",
    "canal",
    "source_file",
    col("cliente.cliente_id").alias("cliente_id"),
    col("cliente.nombre").alias("cliente_nombre"),
    col("cliente.email").alias("cliente_email"),
    col("cliente.direccion.ciudad").alias("cliente_ciudad"),
    col("cliente.direccion.region").alias("cliente_region"),
    col("cliente.direccion.pais").alias("cliente_pais"),
    "items",
)

df_pedidos_cliente.display()


## Lab 2C: Aplanar el array de items con `explode()`

`items` sí es un array: cada pedido puede tener varios productos. `explode()` convierte cada elemento del array en una fila independiente, cambiando el grano de la tabla de "un pedido" a "un ítem de un pedido". Es una decisión de modelado, no solo una operación técnica: a partir de acá, `monto_total` por pedido se recalcula agregando, no leyendo una sola columna.

In [ ]:
from pyspark.sql.functions import explode, current_timestamp, lit

df_items = (
    df_pedidos_cliente
    .withColumn("item", explode(col("items")))
    .select(
        "pedido_id",
        "fecha_pedido",
        "canal",
        "source_file",
        "cliente_id",
        "cliente_nombre",
        "cliente_email",
        "cliente_ciudad",
        "cliente_region",
        "cliente_pais",
        col("item.producto").alias("producto"),
        col("item.categoria").alias("categoria"),
        col("item.cantidad").alias("cantidad"),
        col("item.precio_unitario").alias("precio_unitario"),
    )
    .withColumn("monto_item", col("cantidad") * col("precio_unitario"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("ecommerce_pedidos"))
    .withColumn("batch_id", lit("carga_json_2026-06"))
)

df_items.write.mode("overwrite").saveAsTable("dbassociate.default.pedidos_ecommerce_lab2")

print("Filas en la tabla plana (una por ítem de pedido):", df_items.count())


## Lab 2D: Validar el resultado

In [ ]:
spark.sql("""
    SELECT cliente_ciudad, categoria, COUNT(*) AS num_items, ROUND(SUM(monto_item), 2) AS monto_total
    FROM dbassociate.default.pedidos_ecommerce_lab2
    GROUP BY cliente_ciudad, categoria
    ORDER BY monto_total DESC
    LIMIT 10
""").show(truncate=False)


## Limpieza

In [ ]:
spark.sql("DROP TABLE IF EXISTS dbassociate.default.pedidos_ecommerce_lab2")

print("Tabla temporal de este laboratorio eliminada.")
